In [ ]:
from gerrychain import Graph, Election, Partition, MarkovChain, constraints
from gerrychain.updaters import cut_edges, Tally
from gerrychain.accept import always_accept
from gerrychain.proposals import recom
import matplotlib.pyplot as plt
from functools import partial
import numpy as np

graph_fl = Graph.from_file("./PA/PA.shp")

In [ ]:
# Total population in the graph
tot_pop = sum(graph_fl.nodes[v]['TOTPOP'] for v in graph_fl.nodes())
tot_pop

In [ ]:
# Define the 2024 presidential election
pres24 = Election("PRES24", {"Democratic" : "G24PREDHAR", "Republican" : "G24PRERTRU"})

# Create the starting partition
initial_partition = Partition(
    graph_fl,
    assignment = "CD",
    updaters = {
        "cut_edges" : cut_edges,
        "population": Tally("TOTPOP", alias="population"),
        "hispanic_population": Tally("HISP", alias="hispanic_population"),
        "black_population": Tally("NH_BLACK", alias="black_population"),
        "PRES24": pres24
    }    
)

initial_partition

In [ ]:
num_dist = 17
ideal_pop = tot_pop/num_dist
pop_tolerance = 0.03
# Set up the ReCom proposal for the random walk
random_walk_prop = partial(
    recom, 
    pop_col="TOTPOP",
    pop_target = ideal_pop,
    epsilon= pop_tolerance,
    node_repeats = 1
)

# Creating the population balance contraint
population_constraint = constraints.within_percent_of_ideal_population(
    initial_partition, 
    pop_tolerance, 
    pop_key = "population"
)

In [ ]:
# Building Markov chain
random_walk_20 = MarkovChain(
    proposal = random_walk_prop,
    constraints = [population_constraint], 
    accept = always_accept,
    initial_state = initial_partition,
    total_steps = 20000
)

random_walk_40 = MarkovChain(
    proposal = random_walk_prop,
    constraints = [population_constraint], 
    accept = always_accept,
    initial_state = initial_partition,
    total_steps = 40000
)

In [ ]:
# Run the chain and collect the edge counts
cut_edges_list_20 = []
efficiency_gap_list_20 = []
dem_wins_list = []
dem_vote_shares = []

black_majority = []

for part in random_walk_20:
    cut_edges_list_20.append(len(part["cut_edges"]))
    efficiency_gap_list_20.append(part["PRES24"].efficiency_gap())
    dem_wins_list.append(part["PRES24"].wins("Democratic"))
    dem_vote_shares.append(sorted(part["PRES24"].percents("Democratic")))

    count = 0

    # Do the same for the other populations
    for district in part.parts:
        black_pop = part["black_population"][district]
        total_pop = part["population"][district]

        if black_pop / total_pop > 0.5:
            count += 1

    black_majority.append(count)

print(cut_edges_list_20)
print(efficiency_gap_list_20)
print(dem_wins_list)
print(dem_vote_shares)

In [ ]:
cut_edges_list_40 = []
efficiency_gap_list_40 = []
dem_wins_list_40 = []
dem_vote_shares_40 = []

black_majority_40 = []

for part in random_walk_40:
    cut_edges_list_40.append(len(part["cut_edges"]))
    efficiency_gap_list_40.append(part["PRES24"].efficiency_gap())
    dem_wins_list_40.append(part["PRES24"].wins("Democratic"))
    dem_vote_shares_40.append(sorted(part["PRES24"].percents("Democratic")))

    count = 0

    # Do the same for the other populations
    for district in part.parts:
        black_pop = part["black_population"][district]
        total_pop = part["population"][district]

        if black_pop / total_pop > 0.5:
            count += 1

    black_majority_40.append(count)


print(cut_edges_list_40)
print(efficiency_gap_list_40)

In [ ]:
plt.figure(figsize=(10, 6))

district_data = np.array(dem_vote_shares).T

plt.boxplot(district_data)
plt.axhline(0.5, linestyle="--")

plt.xlabel("District ranked by Democratic vote share")
plt.ylabel("Democratic vote share")

plt.title("Marginal Box Plot: 2024 Presidential")

plt.show()

In [ ]:
# Plotting the the cut edge counts
plt.figure(figsize=(12, 6))

plt.hist(cut_edges_list_20, align='left')

plt.show()
# Plotting the the cut edge counts
plt.figure(figsize=(12, 6))

plt.hist(cut_edges_list_40, align='left')

plt.show()

In [ ]:
# Define the 2024 General Election (Senate)
sen24 = Election("GEN24", {"Democratic" : "G24USSDCAS", "Republican" : "G24USSRMCC"})

# Create the starting partition
initial_partition2 = Partition(
    graph_fl,
    assignment = "CD",
    updaters = {
        "cut_edges" : cut_edges,
        "population": Tally("TOTPOP", alias="population"),
        "hispanic_population": Tally("HISP", alias="hispanic_population"),
        "black_population": Tally("NH_BLACK", alias="black_population"),
        "GEN24": sen24
    }    
)

initial_partition2

In [ ]:
# Building a short Markov chain
random_walk2 = MarkovChain(
    proposal = random_walk_prop,
    constraints = [population_constraint],
    accept = always_accept,
    initial_state = initial_partition2,
    total_steps = 20000
)

In [ ]:
# Run the chain and collect the edge counts
cut_edges_list2 = []

for part in random_walk2:
    cut_edges_list2.append(len(part["cut_edges"]))

print(cut_edges_list2)

In [ ]:
# Plotting the the cut edge counts for the senate election
plt.figure(figsize=(12, 6))

plt.hist(cut_edges_list2, align='left')

plt.show()

In [ ]:
# Define the 2024 General Election (Attorney General)
atg24 = Election("ATG24", {"Democratic" : "G24ATGDDEP", "Republican" : "G24ATGRSUN"}) 

# Create the starting partition
initial_partition3 = Partition(
    graph_fl,
    assignment = "CD",
    updaters = {
        "cut_edges" : cut_edges,
        "population": Tally("TOTPOP", alias="population"),
        "hispanic_population": Tally("HISP", alias="hispanic_population"),
        "black_population": Tally("NH_BLACK", alias="black_population"),
        "ATG24": atg24
    }    
)

initial_partition3

In [ ]:
# Building a short Markov chain
random_walk3 = MarkovChain(
    proposal = random_walk_prop,
    constraints = [population_constraint],
    accept = always_accept,
    initial_state = initial_partition3,
    total_steps = 20000
)

In [ ]:
# Run the chain and collect the edge count
cut_edges_list3 = []

for part in random_walk3:
    cut_edges_list3.append(len(part["cut_edges"]))

print(cut_edges_list3)

In [ ]:
# Plotting the the cut edge counts for attorney general election
plt.figure(figsize=(12, 6))

plt.hist(cut_edges_list3, align='left')

plt.show()